In [11]:
# ===== Imports =====
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ===== Settings =====
CSV_PATH = Path("pf2315_readings.csv")
FS = 30.0  # sampling rate


# ===== Load CSV =====
df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df)} rows")
print(df.columns.tolist())


# ===== Time Axis =====
# Use sample index if available
if "sample_index" in df.columns:
    df["time_s"] = df["sample_index"] / FS
else:
    df["time_s"] = np.arange(len(df)) / FS


# ===== Acceleration Magnitude =====
df["accel_magnitude"] = np.sqrt(
    df["accel_x_g"]**2 +
    df["accel_y_g"]**2 +
    df["accel_z_g"]**2
)


# =========================================================
# IR SIGNALS
# =========================================================
ir_cols = ["ir1", "ir2", "ir3", "ir4"]

fig_ir = go.Figure()

for col in ir_cols:
    if col in df.columns:
        fig_ir.add_trace(
            go.Scatter(
                x=df["time_s"],
                y=df[col],
                mode="lines",
                name=col,
            )
        )

fig_ir.update_layout(
    title="Raw IR Signals",
    xaxis_title="Time (s)",
    yaxis_title="IR Value",
    hovermode="x unified",
)

fig_ir.show()


# =========================================================
# RED SIGNALS
# =========================================================
red_cols = ["red1", "red2", "red3", "red4"]

fig_red = go.Figure()

for col in red_cols:
    if col in df.columns:
        fig_red.add_trace(
            go.Scatter(
                x=df["time_s"],
                y=df[col],
                mode="lines",
                name=col,
            )
        )

fig_red.update_layout(
    title="Raw Red Signals",
    xaxis_title="Time (s)",
    yaxis_title="Red Value",
    hovermode="x unified",
)

fig_red.show()


# =========================================================
# ACCEL X/Y/Z
# =========================================================
fig_accel = go.Figure()

accel_cols = [
    "accel_x_g",
    "accel_y_g",
    "accel_z_g",
]

for col in accel_cols:
    if col in df.columns:
        fig_accel.add_trace(
            go.Scatter(
                x=df["time_s"],
                y=df[col],
                mode="lines",
                name=col,
            )
        )

fig_accel.update_layout(
    title="Accelerometer Signals",
    xaxis_title="Time (s)",
    yaxis_title="Acceleration (g)",
    hovermode="x unified",
)

fig_accel.show()


# =========================================================
# ACCELERATION MAGNITUDE
# =========================================================
fig_mag = go.Figure()

fig_mag.add_trace(
    go.Scatter(
        x=df["time_s"],
        y=df["accel_magnitude"],
        mode="lines",
        name="Magnitude",
    )
)

fig_mag.update_layout(
    title="Acceleration Magnitude",
    xaxis_title="Time (s)",
    yaxis_title="Magnitude (g)",
    hovermode="x unified",
)

fig_mag.show()

Loaded 2435 rows
['host_time_iso', 'host_time_unix_ns', 'sample_index', 'record_type', 'ir1', 'ir2', 'ir3', 'ir4', 'ir1_repeat', 'ir2_repeat', 'ir3_repeat', 'ir4_repeat', 'red1', 'red2', 'red3', 'red4', 'accel_x_raw', 'accel_y_raw', 'accel_z_raw', 'accel_x_g', 'accel_y_g', 'accel_z_g', 'reserved', 'heart_rate', 'debug']


In [12]:
from scipy.signal import butter, filtfilt, find_peaks, medfilt, resample, find_peaks

def preprocess_ppg_channel(
    raw_signal,
    old_fs=30.0,
    new_fs=100.0,
    lowcut=0.5,
    highcut=6.0,
    order=3,
):
    result = {}

    # =====================================================
    # RAW
    # =====================================================
    raw_signal = np.asarray(raw_signal, dtype=float)

    # Invert if needed
    signal = -raw_signal

    result["raw"] = raw_signal

    # =====================================================
    # RESAMPLE
    # =====================================================
    n_new = int(len(signal) * new_fs / old_fs)

    resampled = resample(signal, n_new)

    result["resampled"] = resampled

    # =====================================================
    # BANDPASS
    # =====================================================
    filtered = bandpass_ppg(
        resampled,
        fs=new_fs,
        low=lowcut,
        high=highcut,
        order=order,
    )

    result["filtered"] = filtered

    # =====================================================
    # VALLEY DETECTION
    # =====================================================
    valleys, peaks, xf = detect_valleys_ppg(
        filtered,
        fs=new_fs,
    )

    result["valleys"] = valleys
    result["peaks"] = peaks

    # =====================================================
    # BASELINE REMOVAL
    # =====================================================
    detrended, baseline, segments = spline_baseline_removal_v2(
        filtered,
        valleys,
    )

    result["baseline"] = baseline
    result["detrended"] = detrended
    result["segments"] = segments
    result["n_segments"] = len(segments)

    # =====================================================
    # HEART RATE
    # =====================================================
    hr = sample_heart_rate(
        detrended,
        fs=new_fs,
    )

    result["hr"] = hr

    return result

def sample_heart_rate(sig, fs):
    min_dist = int(0.4 * fs)

    pks, _ = find_peaks(
        sig,
        distance=min_dist,
        prominence=0.01,
    )

    if len(pks) < 2:
        return float("nan")

    rr = np.diff(pks) / fs

    return round(60.0 / np.mean(rr), 1)

def bandpass_ppg(x, fs, low=0.5, high=8.0, order=3):
    """
    Bandpass for PPG morphology: keeps ~heart-rate band and removes drift/high noise.
    Tune low/high based on your data (fs=125 typical).
    """
    x = np.asarray(x, dtype=np.float32)
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, x)

# def remove_baseline_median(x, fs, win_sec=1.5):
#     k = int(win_sec * fs)
#     if k % 2 == 0: k += 1
#     base = medfilt(x, kernel_size=k)
#     return x - base

def detect_valleys_ppg(
    signal,
    fs=50.0,
    low=0.5,
    high=8.0,
    peak_min_dist_s=0.05,     # min distance between systolic peaks (sec)
    peak_prominence=None,     # if None, auto from robust stats
    valley_search_s=(0.15, 0.7)  # search window BEFORE peak for valley (sec)
    # baseline_win_s=1.5,
    # between_peaks_margin_s=0.08,
    # valley_mad_thresh=6.0
):
    """
    Returns:
      valleys: np.array of valley indices
      peaks: np.array of systolic peak indices
      filt: filtered signal used for detection
    """
    x = np.asarray(signal, dtype=np.float32)

    # if baseline_win_s is not None:
    #     x = remove_baseline_median(x, fs, win_sec=baseline_win_s)

    # 1) Filter
    xf = bandpass_ppg(x, fs=fs, low=low, high=high, order=3)

    # 2) Peak detection (systolic peaks)
    min_dist = int(peak_min_dist_s * fs)

    # Auto prominence: based on MAD (robust)
    if peak_prominence is None:
        med = np.median(xf)
        mad = np.median(np.abs(xf - med)) + 1e-8
        peak_prominence = 3.0 * mad  # adjust 2.0–4.0 depending on noise

    peaks, props = find_peaks(xf, distance=min_dist, prominence=peak_prominence)

    # 3) Valley detection: for each peak, search for minimum in a window before it
    vmin = int(valley_search_s[0] * fs)
    vmax = int(valley_search_s[1] * fs)

    valleys = []
    for p in peaks:
        left = max(0, p - vmax)
        right = max(0, p - vmin)
        if right <= left:
            continue
        seg = xf[left:right]
        # v_local = np.argmin(seg)
        # if seg[v_local] < -6 * mad:
        #   continue
        v = left + int(np.argmin(seg))
        valleys.append(v)

    valleys = np.array(valleys, dtype=int)

    # 4) Cleanup: ensure strictly increasing and unique
    if len(valleys) > 1:
        valleys = np.unique(valleys)

    return valleys, peaks, xf

from scipy.interpolate import CubicSpline

def spline_baseline_removal_v2(signal, valleys):    
    spline = CubicSpline(valleys, signal[valleys], bc_type="natural", extrapolate=False)
    baseline = np.full_like(signal, np.nan, dtype=np.float32)
    
    valid = (np.arange(len(signal)) >= valleys[0]) & \
            (np.arange(len(signal)) <= valleys[-1])

    baseline[valid] = spline(np.arange(len(signal))[valid])

    detrended = signal.copy()
    detrended[valid] -= baseline[valid]
    
    detrended = detrended[valleys[0]:valleys[-1]+1]
    
    segments = []
    for i in range(len(valleys) - 1):
        start = valleys[i]
        end = valleys[i + 1]
        segment = detrended[start - valleys[0]:end - valleys[0] + 1]
        segments.append(segment)
        
    
    
    return detrended, baseline, segments

In [13]:
ir_results = {}
red_results = {}

for col in ["ir1", "ir2", "ir3", "ir4"]:
    ir_results[col] = preprocess_ppg_channel(df[col])

for col in ["red1", "red2", "red3", "red4"]:
    red_results[col] = preprocess_ppg_channel(df[col])

In [14]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=4,
    cols=2,
    subplot_titles=[
        "IR1", "RED1",
        "IR2", "RED2",
        "IR3", "RED3",
        "IR4", "RED4",
    ],
    shared_xaxes=True,
)

# =====================================================
# LOOP THROUGH CHANNELS
# =====================================================
for i in range(1, 5):

    ir_key = f"ir{i}"
    red_key = f"red{i}"

    # -------------------------
    # IR subplot
    # -------------------------
    fig.add_trace(
        go.Scatter(
            y=ir_results[ir_key]["filtered"],
            mode="lines",
            name=f"{ir_key} filtered",
        ),
        row=i,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            y=ir_results[ir_key]["baseline"],
            mode="lines",
            name=f"{ir_key} baseline",
        ),
        row=i,
        col=1,
    )

    # # Optional valleys
    # valleys = ir_results[ir_key]["valleys"]

    # fig.add_trace(
    #     go.Scatter(
    #         x=valleys,
    #         y=ir_results[ir_key]["filtered"][valleys],
    #         mode="markers",
    #         name=f"{ir_key} valleys",
    #     ),
    #     row=i,
    #     col=1,
    # )

    # -------------------------
    # RED subplot
    # -------------------------
    fig.add_trace(
        go.Scatter(
            y=red_results[red_key]["filtered"],
            mode="lines",
            name=f"{red_key} filtered",
        ),
        row=i,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            y=red_results[red_key]["baseline"],
            mode="lines",
            name=f"{red_key} baseline",
        ),
        row=i,
        col=2,
    )

    # valleys = red_results[red_key]["valleys"]

    # fig.add_trace(
    #     go.Scatter(
    #         x=valleys,
    #         y=red_results[red_key]["filtered"][valleys],
    #         mode="markers",
    #         name=f"{red_key} valleys",
    #     ),
    #     row=i,
    #     col=2,
    # )

# =====================================================
# LAYOUT
# =====================================================
fig.update_layout(
    height=1400,
    width=1400,
    title="IR and RED Preprocessing",
    hovermode="x unified",
)

fig.show()

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNBackbone(nn.Module):
  def __init__(self, in_channels=1, out_channels=1):
    super().__init__()
    self.conv1 = nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3)
    self.bn1 = nn.BatchNorm1d(64)

    self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv3 = nn.Conv1d(128, 256, kernel_size=3, stride=2, padding=1)
    self.bn3 = nn.BatchNorm1d(256)

    # self.conv4 = nn.Conv1d(256, 128, kernel_size=3, stride=1, padding=2, dilation=2, bias=False)
    # self.bn4 = nn.BatchNorm1d(128)

    self.emb_dim = 256

  def forward(self, x):
    x = F.relu(self.bn1(self.conv1(x)))
    x = F.relu(self.bn2(self.conv2(x)))
    x = F.relu(self.bn3(self.conv3(x)))
    # x = F.relu(self.bn4(self.conv4(x)))
    return x

def masked_gap_from_zero_padding(features, mask, eps=1e-6):
  """
  features: [B, C, T_feat]
  returns:  [B, C]
  """
  if mask.dim() == 2:
        mask = mask.unsqueeze(1)
  B, C, T_feat = features.shape
  mask_feat = F.interpolate(mask.float(), size=T_feat, mode="area")  # [B,1,T_feat]
  masked = features * mask_feat
  denom = mask_feat.sum(dim=-1).clamp(min=eps)  # [B,1]
  return masked.sum(dim=-1) / denom             # [B,C]

class DemoMLP(nn.Module):
  def __init__(self, in_dim, emb_dim=64, dropout=0.2):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, 64),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(64, emb_dim),
        nn.ReLU(),
    )
    self.emb_dim = emb_dim

  def forward(self, d):
    if d.dim() == 3:
      d = d.squeeze(1)
    return self.net(d)

class MultiModalModel(nn.Module):
  def __init__(self, demo_dim=8, dropout=0.3):
    super().__init__()
    self.cnn = CNNBackbone()
    self.mlp = DemoMLP(demo_dim, emb_dim=256, dropout=dropout)
    fused_dim = self.cnn.emb_dim + self.mlp.emb_dim

    self.head = nn.Sequential(
        nn.Linear(fused_dim, 512),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(256, 64),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(64,1)
    )

  def forward(self, x, m=None, d=None):
    feats = self.cnn(x)
    sig_emb = masked_gap_from_zero_padding(feats, m)
    if d is None:
      fused = sig_emb
    else:
      demo_emb = self.mlp(d)
      # print(sig_emb.shape)
      # print(demo_emb.shape)
      fused = torch.cat([sig_emb, demo_emb], dim=1)

    out = self.head(fused).squeeze(-1)
    return out

In [16]:
from sklearn.preprocessing import MinMaxScaler

def build_model_input(
    processed_result,
    start_segment=0,
    n_segments=15,
    segment_length=100,
):
    """
    Creates:
        signal_1500
        mask_1500

    from selected segments.
    """

    segments = processed_result["segments"]

    # =====================================================
    # SELECT SEGMENTS
    # =====================================================
    selected_segments = segments[
        start_segment:start_segment + n_segments
    ]

    if len(selected_segments) == 0:
        raise ValueError("No segments selected")

    lengths = [len(s) for s in selected_segments]

    # =====================================================
    # CONCATENATE
    # =====================================================
    concat_signal = np.concatenate(selected_segments)

    # =====================================================
    # NORMALIZE
    # =====================================================
    scaler = MinMaxScaler(feature_range=(0, 1))

    normalized = scaler.fit_transform(
        concat_signal.reshape(-1, 1)
    ).flatten()

    # =====================================================
    # SPLIT BACK
    # =====================================================
    normalized_segments = []

    idx = 0

    for length in lengths:
        normalized_segments.append(
            normalized[idx:idx + length]
        )
        idx += length

    # =====================================================
    # PAD/TRUNCATE
    # =====================================================
    padded = []
    masks = []

    for seg in normalized_segments:

        seg = np.asarray(seg)[:segment_length]

        real_len = len(seg)

        pad = segment_length - real_len

        padded.append(
            np.pad(
                seg,
                (0, pad),
                constant_values=0,
            )
        )

        masks.append(
            np.concatenate([
                np.ones(real_len),
                np.zeros(pad),
            ])
        )

    signal_1500 = np.concatenate(padded)
    mask_1500 = np.concatenate(masks)

    return {
        "signal_1500": signal_1500,
        "mask_1500": mask_1500,
        "n_segments_used": len(selected_segments),
    }

In [17]:
import pandas as pd
import numpy as np
import torch
import joblib
from pathlib import Path

scaler_path = Path("../glucosense/scaler.pkl")

std_scaler = joblib.load(scaler_path)


def build_demo_tensor(
    age,
    weight,
    height,
    sex,
    preop_dm,
    preop_htn,
    hr,
):
    # =====================================================
    # BMI
    # =====================================================
    bmi = weight / ((height / 100) ** 2)

    # fallback HR
    if np.isnan(hr):
        hr = 70

    # =====================================================
    # MATCH TRAINING FORMAT EXACTLY
    # =====================================================
    cont = pd.DataFrame(
        [[
            age,
            weight,
            bmi,
            height,
            hr,
        ]],
        columns=[
            "age",
            "weight",
            "bmi",
            "height",
            "actual_hr",
        ]
    )

    # =====================================================
    # SCALE
    # =====================================================
    age_s, weight_s, bmi_s, height_s, hr_s = (
        std_scaler.transform(cont)[0]
    )

    # =====================================================
    # BUILD TENSOR
    # =====================================================
    demo = torch.tensor([
        [
            age_s,
            height_s,
            weight_s,
            bmi_s,
            hr_s,
            sex,
            preop_dm,
            preop_htn,
        ]
    ], dtype=torch.float32)

    return demo

In [18]:
def run_inference(
    signal_1500,
    mask_1500,
    demo,
    model_path="../glucosense/best_model.pt",
):
    model = MultiModalModel()

    model.load_state_dict(
        torch.load(
            model_path,
            map_location="cpu",
        )
    )

    model.eval()

    x = torch.tensor(
        signal_1500,
        dtype=torch.float32,
    ).unsqueeze(0).unsqueeze(0)

    m = torch.tensor(
        mask_1500,
        dtype=torch.float32,
    ).unsqueeze(0).unsqueeze(0)

    with torch.no_grad():

        prediction = model(
            x,
            m=m,
            d=demo,
        ).item()

    return prediction

In [60]:
selected_signal = ir_results["ir2"]

model_input = build_model_input(
    selected_signal,
    start_segment=10,
    n_segments=15,
)

signal_1500 = model_input["signal_1500"]
mask_1500 = model_input["mask_1500"]

In [61]:
demo = build_demo_tensor(
    age=24,
    weight=53,
    height=170,
    sex=1,
    preop_dm=0,
    preop_htn=0,
    hr=selected_signal["hr"],
)

In [62]:
prediction = run_inference(
    signal_1500,
    mask_1500,
    demo,
)

print("Prediction:", prediction)

Prediction: 110.9176254272461


In [63]:
# plot the signal 1500 with mask
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        y=signal_1500,
        mode="lines",
        name="Signal 1500",
    )
)

fig.add_trace(
    go.Scatter(
        y=mask_1500,
        mode="lines",
        name="Mask 1500",
    )
)

fig.update_layout(
    title="Model Input Signal and Mask",
    xaxis_title="Time Steps",
    yaxis_title="Value",
    hovermode="x unified",
)

fig.show()